## インポート

In [2]:
import scraping
import create_rawdf
import pandas as pd
from bs4 import BeautifulSoup
import pickle
from tqdm import tqdm
from pathlib import Path
import create_prediction_population
DATA_DIR = Path("..", "data")
HTML_DIR = DATA_DIR / "html"
HTML_RACE_DIR = HTML_DIR / "race"
DATA_RAWDF = DATA_DIR / "rawdf"
%load_ext autoreload

## race_idの取得

In [ ]:
import scraping

# レース開催日時を取得後、レースIDを取得
kaisai_date_list = scraping.scrape_kaisai_date(from_= "2017-01", to_ = "2024-12")
# kaisai_date_list = scraping.scrape_kaisai_date(from_= "2024-09", to_ = "2024-10")
race_id_list = scraping.scrape_race_id_list(kaisai_date_list)
len(race_id_list)

100%|██████████| 2/2 [00:03<00:00,  1.60s/it]


['20240901', '20240907', '20240908', '20240914', '20240915', '20240916', '20240921', '20240922', '20240928', '20240929', '20241005', '20241006', '20241012', '20241013', '20241014', '20241019', '20241020', '20241026', '20241027']


100%|██████████| 19/19 [01:20<00:00,  4.26s/it]


540

In [6]:
# race_id一覧の保存
with open("race_id_list.pickle", "wb") as f:
    pickle.dump(race_id_list, f)

## レースページの取得

In [10]:

html_paths_race = []
for race_id in race_id_list:
    file_path = HTML_RACE_DIR / f"{race_id}.bin"
    html_paths_race.append(file_path)
len(html_paths_race)

540

In [7]:
import pickle
import scraping

with open("race_id_list.pickle", "rb") as f:
    race_id_list = pickle.load(f)
len(race_id_list)
html_paths_race = scraping.scrape_html_race(race_id_list=race_id_list)

100%|██████████| 540/540 [00:00<00:00, 52214.86it/s]

skipped : 202404030801
skipped : 202404030802
skipped : 202404030803
skipped : 202404030804
skipped : 202404030805
skipped : 202404030806
skipped : 202404030807
skipped : 202404030808
skipped : 202404030809
skipped : 202404030810
skipped : 202404030811
skipped : 202404030812
skipped : 202407020801
skipped : 202407020802
skipped : 202407020803
skipped : 202407020804
skipped : 202407020805
skipped : 202407020806
skipped : 202407020807
skipped : 202407020808
skipped : 202407020809
skipped : 202407020810
skipped : 202407020811
skipped : 202407020812
skipped : 202401020801
skipped : 202401020802
skipped : 202401020803
skipped : 202401020804
skipped : 202401020805
skipped : 202401020806
skipped : 202401020807
skipped : 202401020808
skipped : 202401020809
skipped : 202401020810
skipped : 202401020811
skipped : 202401020812
skipped : 202406040101
skipped : 202406040102
skipped : 202406040103
skipped : 202406040104
skipped : 202406040105
skipped : 202406040106
skipped : 202406040107
skipped : 2

## レース結果のテーブルをすべて結合 ->race.csv

In [ ]:
# #5401, 6621, 7607, 8250, 9407, 13969, 21687, 22649, 24999 でエラー
import scraping
import create_rawdf
# html_paths_race = list(scraping.HTML_RACE_DIR.glob("*.bin"))
# len(html_paths_race)
# html_paths_race[5401]
results = create_rawdf.create_results(html_path_list=html_paths_race)

  2%|▏         | 13/540 [00:01<00:49, 10.58it/s]

tabele not found at 202407020801


100%|██████████| 540/540 [00:50<00:00, 10.64it/s]


In [14]:
results.reset_index()[["race_id", "horse_id"]].duplicated().sum()

np.int64(0)

## race.csvからhorse_idをすべて取得、スクレイピング

In [1]:
import pandas as pd
from pathlib import Path

path = Path("..", "data", "rawdf", "results.csv")
results = pd.read_csv(path, sep="\t")
horse_id_list = results["horse_id"].unique()
len(horse_id_list)

44929

In [6]:
import scraping
scraping.scrape_html_horse(horse_id_list=horse_id_list)

100%|██████████| 44091/44091 [00:18<00:00, 2409.19it/s] 


[PosixPath('../data/html/horse_java/2013103391.bin'),
 PosixPath('../data/html/horse_java/2021103472.bin')]

## スクレイピング済みhorse_id一覧をpickleファイルに保存
容量圧迫していたらすべてdeleteしてこのpickleファイルを参照->prediction時でなかったらとばすという処理に変更

In [7]:
import os
import pickle

# ディレクトリのパス
directory = "../data/html/horse_java"

# pickleファイルの出力先
output_pickle = "horse_id.pickle"

# .binファイルの名前を収集
filenames = []
for file in os.listdir(directory):
    if file.endswith(".bin"):
        # 拡張子を除いた名前を取得
        filenames.append(os.path.splitext(file)[0])

# pickleファイルに書き込む
with open(output_pickle, "wb") as pickle_file:
    pickle.dump(filenames, pickle_file)

print(f"{len(filenames)}個のファイル名を {output_pickle} に書き込みました。")


44091個のファイル名を horse_id.pickle に書き込みました。


In [9]:
# pickleファイルの確認
with open("horse_id.pickle", "rb") as pickle_file:
    filenames = pickle.load(pickle_file)

len(filenames)

44091

## 馬ページのレース結果をすべて取得、結合 ->horse_results.csv

In [4]:
# パスをすべて取得
import scraping
html_paths_horse = list(scraping.HTML_HORSE_JAVA_DIR.glob("*.bin"))
len(html_paths_horse)
horse_ids_from_paths = set(path.stem for path in html_paths_horse)  # ファイル名から拡張子を除去して取得

# horse_id_list に含まれていない horse_id を計算
missing_horse_ids = horse_ids_from_paths - horse_id_list
len(missing_horse_ids)

187

In [3]:
import pandas as pd
from pathlib import Path

path = Path("..", "data", "rawdf", "horse_results.csv")
results = pd.read_csv(path, sep="\t")
horse_id_list = set(results["horse_id"].astype(str).unique()) 
# horse_id_list = results["horse_id"].unique()
len(horse_id_list)

/tmp/ipykernel_868389/4198114664.py:5: DtypeWarning: Columns (29,30,31,32) have mixed types. Specify dtype option on import or set low_memory=False.
  results = pd.read_csv(path, sep="\t")


44092

In [ ]:
import create_rawdf
save_dir = scraping.HTML_HORSE_JAVA_DIR

paths = []
for horse_id in missing_horse_ids:
    file_path = save_dir / f"{horse_id}.bin"
    paths.append(file_path)

horse_results = create_rawdf.create_horse_results(paths)

In [ ]:
import create_rawdf

# 表の作成
horse_results = create_rawdf.create_horse_results(html_paths_horse)
horse_results

## 馬情報テーブルの作成

In [ ]:
import create_rawdf
horse_info = create_rawdf.create_horse_info(html_path_list=html_paths_horse)

## レース情報テーブルの作成 -> race_info.csv

In [ ]:
DATA_DIR = Path("..", "data")
HTML_DIR = DATA_DIR / "html"
HTML_RACE_DIR = HTML_DIR / "race"
# html_paths_race = list(HTML_RACE_DIR.glob("*.bin")) # すべてのレースのHTMLを取得
len(html_paths_race)
race_info = create_rawdf.create_race_info(html_paths_race)
# html_path_list[24998]

## 予測時の表作成

In [4]:
population_df = create_prediction_population.create(kaisai_date="20241222")

scraping race_id_list ...


100%|██████████| 1/1 [00:09<00:00,  9.55s/it]


race_id list : ['202406050801', '202406050802', '202406050803', '202406050804', '202406050805', '202406050806', '202406050807', '202406050808', '202406050809', '202406050810', '202406050811', '202406050812', '202408070801', '202408070802', '202408070803', '202408070804', '202408070805', '202408070806', '202408070807', '202408070808', '202408070809', '202408070810', '202408070811', '202408070812']
scraping horse_id_list ...


100%|██████████| 24/24 [00:38<00:00,  1.60s/it]


In [4]:
horse_id_list = population_df["horse_id"].unique()

In [5]:
html_paths_horse = scraping.scrape_html_horse(horse_id_list=horse_id_list, skip = False)

 78%|███████▊  | 287/369 [43:45<10:12,  7.47s/it] 

File too small, retrying: 2020100230


 82%|████████▏ | 301/369 [45:17<06:27,  5.70s/it]

File too small, retrying: 2020102664


 91%|█████████ | 335/369 [50:17<06:01, 10.64s/it]

...

 75%|███████▌  | 30/40 [04:23<01:05,  6.51s/it]

File too small, retrying: 2020100230


 20%|██        | 1/5 [00:03<00:13,  3.26s/it]

File too small, retrying: 2022104017


100%|██████████| 1/1 [00:05<00:00,  5.11s/it]


In [6]:
len(html_paths_horse)

369

In [9]:
# html_paths_horse[8]あかん
horse_results = create_rawdf.create_horse_results(
    html_path_list = html_paths_horse,
    save_filename="horse_results_prediction.csv"
    )

  7%|▋         | 27/369 [00:13<02:29,  2.29it/s]

table not found at 2022107359


  8%|▊         | 28/369 [00:13<02:27,  2.31it/s]

table not found at 2022102555


  8%|▊         | 29/369 [00:13<02:27,  2.31it/s]

...

 96%|█████████▌| 354/369 [02:42<00:05,  2.70it/s]

table not found at 2022104306


 96%|█████████▋| 356/369 [02:43<00:04,  2.64it/s]

table not found at 2022104385


100%|██████████| 369/369 [02:48<00:00,  2.19it/s]


In [10]:
horse_info = create_rawdf.create_horse_info(
    html_path_list = html_paths_horse,
    save_filename="horse_info.csv"
)

100%|██████████| 369/369 [03:15<00:00,  1.89it/s]
